In [ ]:
import jax
import os

In [ ]:
import sys
sys.path.append("../")

%load_ext autoreload
%autoreload 2

import pulse_simulator as ps

In [ ]:
from datetime import datetime
import functools
import itertools
import logging
import numpy as np
import pickle as pkl
import qiskit as qk
from qiskit import pulse
import qiskit_aer
import qiskit.providers.fake_provider as qk_fp
import qiskit_dynamics as qk_d
import qutip as qt
import re 
from scipy.integrate import simpson
from scipy.linalg import sqrtm, expm
from tqdm import tqdm
import warnings

import matplotlib.pyplot as plt

In [ ]:
Units = 1e9

# Undo units
GHz = 1 / Units
MHz = 1e3 / Units
μs = Units / 1e3
ns = Units

backend = qk_fp.FakeManila()
dt = backend.configuration().dt * ns
desired_1qb_duration = 225  # dt
desired_2qb_duration = 225 * 4  # dt

# Gate time in SI units (granularity restriction)
duration_1qb = ps.qiskit_closest_multiple_of_16(desired_1qb_duration * dt / ns, backend)
duration_2qb = ps.qiskit_closest_multiple_of_16(desired_2qb_duration * dt / ns, backend)

# Calibrate

In [ ]:
# Load shapes
fname_1qb = "../pico-pulses/saved-pulses-2024-02-16/a_single_qubit_gateset_R1e-5.csv"
pico_a = ps.from_csv(fname_1qb)
pico_a = np.vstack([pico_a, 1.0 * np.ones(pico_a.shape[1])])

names_1qb = ["X1", "X2", "SX1", "SX2", "I1", "I2"]
angles_1qb = [1.5 * np.pi, 0.5 * np.pi, 1.25 * np.pi, 0.25 * np.pi, 0.2, 0.0]

In [ ]:
cal_1qb_robust = ps.calibrate_gateset(
    gateset=pico_a,
    names=names_1qb,
    angles=angles_1qb,
    duration=duration_1qb,
    dt=dt,
)

In [ ]:
cal_1qb_default = ps.default_single_qubit_gateset(
    names_1qb,
    duration_1qb,
    dt,
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3))

ax = axes[0]
for i, (k, v) in enumerate(cal_1qb_robust.items()):
    ax.plot(ps.get_samples(v).real, label=names_1qb[i])
ax.legend(loc="upper left")

ax = axes[1]
for i, (k, v) in enumerate(cal_1qb_default.items()):
    ax.plot(ps.get_samples(v).real, label=names_1qb[i])
ax.legend(loc="upper left")

In [ ]:
# Load shapes
def fname(k):
    return f"../pico-pulses/saved-pulses-2024-01-16/a_two_qubit_{k}.csv"

base_2qb = {k: ps.from_csv(fname(k))[0] for k in ["default", "robust"]}
robust_2qb = 2 * [base_2qb["robust"]]
default_2qb = 2 * [base_2qb["default"]]
names_2qb = ["+", "-"]

In [ ]:
cal_2qb_robust = ps.calibrate_gateset(
    gateset=robust_2qb,
    names=names_2qb,
    angles=[0.125 * np.pi, -0.125 * np.pi],
    duration=duration_2qb,
    dt=dt,
    samples=32,
)

In [ ]:
cal_2qb_default = ps.calibrate_gateset(
    gateset=default_2qb,
    names=names_2qb,
    angles=[0.125 * np.pi, -0.125 * np.pi],
    duration=duration_2qb,
    dt=dt,
    samples=32,
)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(5, 3))
for i, (k, v) in enumerate(cal_2qb_robust.items()):
    ax.plot(ps.get_samples(v).real, label=names_2qb[i])
ax.legend(loc="upper right")

# Build circuit

In [ ]:
registers = [0, 1]
coupling_map = ps.backend_edges(backend)

xy4_depth = 40

In [ ]:
circ = qk.QuantumCircuit(2, 2)

In [ ]:
def prepare_state(circ, registers):
    [circ.h(r) for r in registers]

def apply_xy4(circ, q, reverse=False):
    if reverse:
        circ.y(q)
        circ.x(q)
        circ.y(q)
        circ.x(q)
    else:
        circ.x(q)
        circ.y(q)
        circ.x(q)
        circ.y(q)

def apply_simultaneous_xy4(circ, q0, q1):
    apply_xy4(circ, q0)
    apply_xy4(circ, q1, reverse=True)

In [ ]:
circ = qk.QuantumCircuit(2, 2)

prepare_state(circ, registers)
for _ in range(xy4_depth):
    apply_simultaneous_xy4(circ, 0, 1)
prepare_state(circ, registers)

# circ.draw('mpl')

In [ ]:
mb = ps.MomentBuilder(
    coupling_map=ps.backend_edges(backend),
    gate_durations={1: duration_1qb, 2: duration_2qb}
)

moments = mb.build_moments(
    circ, 
    optimization_level=0, 
    
)

In [ ]:
sched = ps.build_pulse_schedule(
    moments,
    qubit_gates=cal_1qb_default,
    cr_gates=cal_2qb_default,
    registers=registers,
    backend=backend,
    name="default-hamiltonian-simulation",
    dt=dt,
    insert_identities=True,
    verbose=True
)

robust_sched = ps.build_pulse_schedule(
    moments,
    qubit_gates=cal_1qb_robust,
    cr_gates=cal_2qb_robust,
    registers=registers,
    backend=backend,
    name="robust-hamiltonian-simulation",
    dt=dt,
    insert_identities=True,
    verbose=False
)

# Check durations
assert robust_sched.duration == sched.duration

# Would be nice to disable phase channels when plotting
# sched.draw()

In [ ]:
# circ2 = ps.build_circuit(moments, registers, insert_identities=False, barriers=True)
# circ2.draw('mpl')

# Models

In [ ]:
scale = 1.0

# Set registers above with circuit
# The rest is automatic
# ===================== 

# NOTE: If the Rabi rates are True, you have to calibrate!
cvars = ps.backend_simulation_vars(backend, rabi=False, units=Units)

# Change T1 and T2 to match Eagle processors like Sherbrooke
for k, v in cvars.items():
    if "t1" in k:
        cvars[k] = 250 / μs * ns
    if "t2" in k:
        cvars[k] = 150 / μs * ns

# Carrier frequencies of each control line
# NOTE: We won't use carriers in this example
carriers = ps.backend_carriers(backend, cvars)

# Partially compile model functions with parameters
# =====
H_rx = functools.partial(
    ps.rx_model,
    registers=registers,
    backend=backend,
    variables=cvars,
    rotating_frame=True,
)

H_rz = functools.partial(
    ps.rz_model,
    registers=registers,
    backend=backend,
)

H_cr = functools.partial(
    ps.cross_resonance_model,
    registers=registers,
    backend=backend,
    variables=cvars,
    model_name="Toy",
)

A_decay = functools.partial(ps.qubit_decay_model, registers=registers, variables=cvars)

edges = ps.backend_edges(backend)
print(f"Edges: {edges}")
H_xtalk = scale * ps.crosstalk_model(registers, edges, cvars)

# Create a system model
# =====
H_drift = 0.0
Hs_control = []
Hs_channels = []
As_static = []
for qubit in registers:
    # Single qubit terms
    Hj_drift, Hjs_control, Hjs_channel = H_rx(qubit)
    Hj_phase, Hj_phase_ch = H_rz(qubit)
    Ajs_static = A_decay(qubit)

    H_drift += Hj_drift
    Hs_control += Hjs_control + [Hj_phase]
    Hs_channels += Hjs_channel + [Hj_phase_ch]
    As_static += Ajs_static

    # Two qubit with ring connectivity
    if qubit < len(registers) - 1:
        control, target = qubit, qubit + 1
        
        # Get both directions
        for pair in [(control, target), (target, control)]:
            Hj_drift, Hjs_control, Hjs_channel = H_cr(pair)
            H_drift += Hj_drift
            Hs_control += Hjs_control
            Hs_channels += Hjs_channel

# Some models may repeat control channels
# NOTE: This actually really matters! Repeat channels mess things up.
Hs_channels, indices = np.unique(Hs_channels, return_index=True)
Hs_control = [Hs_control[i] for i in indices]

# For RWA / Rotating frame
# NOTE: We won't use carriers in this example
# =====
registered_carriers = {c: 0 for c in Hs_channels}
cutoff_freq = None

# registered_carriers = {c: carriers[c] for c in Hs_channels}
# cutoff_freq = 2 * np.mean(list(registered_carriers.values()))

# Solvers

In [ ]:
solver_r0 = qk_d.Solver(
    static_hamiltonian=None,
    hamiltonian_operators=Hs_control,
    hamiltonian_channels=Hs_channels,
    rotating_frame=None,
    rwa_cutoff_freq=None,
    channel_carrier_freqs=registered_carriers,
    dt=dt
)

solver_rx0 = qk_d.Solver(
    static_hamiltonian=H_xtalk,
    hamiltonian_operators=Hs_control,
    hamiltonian_channels=Hs_channels,
    rotating_frame=None,
    rwa_cutoff_freq=None,
    channel_carrier_freqs=registered_carriers,
    dt=dt
)

solver_rx = qk_d.Solver(
    static_hamiltonian=H_xtalk,
    hamiltonian_operators=Hs_control,
    hamiltonian_channels=Hs_channels,
    static_dissipators=As_static,
    rotating_frame=None,
    rwa_cutoff_freq=None,
    channel_carrier_freqs=registered_carriers,
    dt=dt
)

In [ ]:
for m in moments:
    if ps.get_moment_case(m) == "phase":
        for q in registers:
            if q not in m.keys():
                m[q] = 0.0

# Runs

In [ ]:
# measure after each non-phase moment
measure_times = dt * np.cumsum(
    ps.durations_of_moments(moments, duration_1qb, duration_2qb)
)#[[ps.get_moment_case(m) != "phase" for m in moments]]
# and at the end time...
# measure_times = np.append(measure_times, sched.duration * dt)


# targets at measure times
Uts = []
for i, m in enumerate(moments):
    # if ps.get_moment_case(m) != "phase":
    Uts.append(ps.build_operator(moments[0:i+1], registers))
# and at the end time...
# Uts.append(ps.build_operator(moments[:-1], registers))

# Target
target = ps.build_operator(moments[:-1], registers)

In [ ]:
# data = np.isclose((measure_times[1:] - measure_times[:-1] ) / dt, 1)
# data[::2]
# data[1::2]


## Without decoherence

In [ ]:
# y0 = ps.qiskit_identity_operator(len(registers))
y0 = ps.qiskit_ground_state(len(registers), open=False)

labels = ["robust",  "default"]
all_schedules = [robust_sched, sched]

# ==========

solver_rx0.model.evaluation_mode = "dense"
res_x0 = solver_rx0.solve(
    t_span=[0.0, sched.duration * dt],
    t_eval=measure_times,
    y0=y0,
    signals=all_schedules,
    atol=1e-9,
    rtol=1e-9,
    method="jax_odeint",
)

# Compute fidelity (we can use our circuit to compute the target)
target = qk.quantum_info.Operator(ps.build_circuit(moments, registers))
y_target = y0.evolve(target)

# for i, ell in enumerate(labels):
#     print(f"\tInfidelity of {ell}: ", 1 - qk.quantum_info.process_fidelity(res_x0[i].y[-1], target))

for i, ell in enumerate(labels):
    print(f"\tInfidelity of {ell} schedule: {1 - qk.quantum_info.state_fidelity(res_x0[i].y[-1], y_target)}")

In [ ]:
fig, axes = plt.subplots(1, figsize=[7, 4])
axes = axes if isinstance(axes, np.ndarray) else [axes]

ax = axes[0]
ax.set_title("XY4: Crosstalk, No decoherence", loc='left')

for i, res in enumerate(res_x0):
    infids = []
    for r, t in zip(res.y, Uts):
        # infids.append(1 - qk.quantum_info.process_fidelity(r, t))
        infids.append(1 - qk.quantum_info.state_fidelity(r, y0.evolve(t)))
    ax.plot(measure_times, infids, label=labels[i], marker=".")

ax.set_xlabel("Time")
ax.legend(loc="upper left")
# ax.set_yscale("log")
ax.set_ylabel("(State?) infidelity")
ax.grid(True)

### Aside

In [ ]:
infids = np.array(infids)
np.argmax(infids[1:] - infids[:-1])

In [ ]:
np.round(res.y[54:57], 1)

In [ ]:
np.round([phi*y0.evolve(t) for phi, t in zip([-1j, 1j, -1], Uts[54:57])], 1)


In [ ]:
for a,b in zip(res.y[54:58], [y0.evolve(t) for t in Uts[54:58]]):
    print(qk.quantum_info.state_fidelity(a, b))

In [ ]:
moments[52:58]

In [ ]:
qk.quantum_info.state_fidelity(r, y0.evolve(t))

## Decoherence

In [ ]:
print("Case 2: Crosstalk and Decoherence")

# y0 = ps.qiskit_identity_operator(len(registers), open=True)
y0 = ps.qiskit_ground_state(len(registers), open=True)

labels = ["robust", "default"]
schedules = [robust_sched, sched]

solver_rx.model.evaluation_mode = "dense"
res_x = solver_rx.solve(
    t_span=[0.0, sched.duration * dt],
    t_eval=measure_times,
    y0=y0,
    signals=schedules,
    atol=1e-9,
    rtol=1e-9,
    method="jax_odeint",
)

# Compute fidelity (we can use our circuit to compute the target)
target = qk.quantum_info.Operator(ps.build_circuit(moments, registers))
y_target = y0.evolve(target)

# for i in range(len(res_x)):
#     print(
#         f"\tInfidelity of {labels[i]} schedule: {1 - qk.quantum_info.process_fidelity(res_x[0].y[-1], target)}"
#     )

for i in range(len(res_x)):
    print(
        f"\tInfidelity of {labels[i]} schedule: {1 - qk.quantum_info.state_fidelity(res_x[i].y[-1], y_target)}"
    )

In [ ]:
fig, axes = plt.subplots(1, figsize=[7, 4])
axes = axes if isinstance(axes, np.ndarray) else [axes]

ax = axes[0]
ax.set_title("XY4: Crosstalk, decoherence", loc='left')


for i, res in enumerate(res_x):
    infids = []
    for r, t in zip(res.y, Uts):
        infids.append(1 - qk.quantum_info.state_fidelity(r, y0.evolve(t)))
    ax.plot(measure_times, infids, label=labels[i], marker=".")

ax.set_xlabel("Time")
ax.legend(loc="upper left")
# ax.set_yscale("log")
ax.set_ylabel("State infidelity")
ax.grid(True)